### Import Libraries

In [3]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import math

In [4]:
# from google.colab import drive
# drive.mount('/content/drive')

In [15]:
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"Using device: {device}")

Using device: mps


### LLama

In [6]:
class RMSNorm(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.gamma = nn.Parameter(torch.ones(config.n_embd)) # (C,)
    self.eps = config.eps

  def forward(self, x):
    B, T, C = x.shape
    rms = torch.sqrt(torch.mean(x**2, dim=-1, keepdim=True) + self.eps) # (B, T, 1)
    x = x/rms # (B, T, C)
    # the broadcast works because PyTorch aligns from the right
    return x * self.gamma # scale features (B, T, C) * (C,) => (B, T, C)

In [7]:
class RotaryEmbedding(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.head_size = config.n_embd // config.n_heads
    assert (config.n_embd // config.n_heads) % 2 == 0, "RoPE requires an even dimension"
    self.block_size = config.block_size
    self.theta = 1.0/10000**(torch.arange(0,self.head_size,2)/self.head_size) # (head_size//2,)
    self.positions = torch.arange(self.block_size) # (block_size,)
    self.freqs = torch.outer(self.positions, self.theta) # (block_size, head_size//2)
    self.register_buffer("cos_cached", torch.cos(self.freqs)) # no gradients + moves to device automatically (all instances of nn.Module)
    self.register_buffer("sin_cached", torch.sin(self.freqs))

  def forward(self, x, start_pos=0):
    B, T, head_size = x.shape
    cos = self.cos_cached[start_pos:start_pos+T] # slices rows => (T, head_size//2) (T is 1 during decode)
    sin = self.sin_cached[start_pos:start_pos+T] # slices rows => (T, head_size//2)
    cos = torch.repeat_interleave(cos, 2, dim=-1) # (T, head_size)
    sin = torch.repeat_interleave(sin, 2, dim=-1) # (T, head_size)
    # (x1, x2) => cos, (-x2, x1) => sin
    return x*cos + self.rotate_half(x)*sin # (B, T, head_size)

  def rotate_half(self, x):
    B, T, head_size = x.shape
    out = torch.zeros_like(x)
    out[...,::2] = -x[...,1::2] # x2 (B, T, head_size//2)
    out[...,1::2] = x[...,::2] # x1 (B, T, head_size//2)
    return out # (B,T,head_size)

In [8]:
class FeedForward(nn.Module):
  def __init__(self, config):
    super().__init__()
    n_embd = config.n_embd
    self.dropout = config.dropout
    hidden = int((8/3)*n_embd)
    self.w1 = nn.Linear(n_embd, hidden, bias=False) # gate
    self.w2 = nn.Linear(n_embd, hidden, bias=False) # info
    self.w3 = nn.Linear(hidden, n_embd, bias=False)
    self.Dropout = nn.Dropout(self.dropout)

  def forward(self, x):
    gate = self.w1(x) # (B,T,C) @ (hidden, C).T => (B,T,hidden)
    info = self.w2(x) # (B,T,hidden)
    out = F.silu(gate)*info # (B,T,hidden)
    out = self.w3(out) # (B,T,hidden)@(hidden, n_embd) => (B,T,n_embd)
    return self.Dropout(out)

#### Grouped Query Attention

    group 0:
      Q0 × K0 → softmax → weighted sum of V0
      Q1 × K0 → softmax → weighted sum of V0
      Q2 × K0 → softmax → weighted sum of V0
      Q3 × K0 → softmax → weighted sum of V0




Example

2 * n_kv_heads * head_size = 2 * 2 * 4 = 16

output last dim: [k0,k1,k2,k3,k4,k5,k6,k7, v0,v1,v2,v3,v4,v5,v6,v7]


In [9]:
class MultiHeadAttention(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.n_heads = config.n_heads
    self.n_kv_heads = config.n_kv_heads # total no. of kv heads (groups)
    self.n_groups = self.n_heads//self.n_kv_heads # no. of q heads per group (group size)
    self.head_size = config.n_embd // self.n_heads
    self.flash = hasattr(F, "scaled_dot_product_attention") # check availability (module, func name)
    self.block_size = config.block_size
    self.n_embd = config.n_embd
    self.dropout = config.dropout

    # GQA projections
    self.q_proj = nn.Linear(self.n_embd, self.n_heads*self.head_size, bias=False) # (n_embd, n_embd)
    self.kv_proj = nn.Linear(self.n_embd, 2*self.n_kv_heads*self.head_size, bias=False) # < n_embd
    self.out_proj = nn.Linear(self.n_embd, self.n_embd, bias=False)
    self.Dropout = nn.Dropout(self.dropout)

    self.rope = RotaryEmbedding(config)
    self.k_cache = None # (B, n_kv_heads, T_cached, head_size)
    self.v_cache = None
    self.register_buffer("tril", torch.tril(torch.ones(self.block_size, self.block_size)).view(1, 1, self.block_size, self.block_size)) # broadcast => (B, n_kv_heads, T, T)

  def apply_rope(self, x, start_pos=0):
    # x: q or k
    B, n_heads, T, head_size = x.shape # (B, n_heads, T, head_size)
    x = x.reshape(B*n_heads, T, head_size)
    x = self.rope(x, start_pos=start_pos)
    x = x.reshape(B, n_heads, T, head_size)
    return x

  def clear_cache(self):
    self.k_cache = None
    self.v_cache = None

  def forward(self, x, use_cache=False):
    B, T, C = x.shape
    # Q is never cached (you don't need previous queries, only previous K and V).
    q = self.q_proj(x) # (B,T,C) @ (C,C) => (B,T,C) # if decode: calc for the current input (T=1)
    k, v = self.kv_proj(x).split(self.n_kv_heads*self.head_size, dim=-1) # both (B, T, kv_heads*head_size); kv_heads*head_size < n_embd since n_kv_heads < n_heads

    # split into heads
    q = q.view(B, T, self.n_heads, self.head_size).transpose(1, 2) # (B, n_heads, T, head_size)
    k = k.view(B, T, self.n_kv_heads, self.head_size).transpose(1, 2) # (B, n_kv_heads, T, head_size)
    v = v.view(B, T, self.n_kv_heads, self.head_size).transpose(1, 2) # (B, n_kv_heads, T, head_size)

    # rope
    start_pos = 0 if self.k_cache is None else self.k_cache.shape[2]
    start_pos = start_pos % self.block_size
    q = self.apply_rope(q, start_pos)
    k = self.apply_rope(k, start_pos)

    # kv cache
    if use_cache: # used during inference
      if self.k_cache is None:
        self.k_cache = k # (B, n_kv_heads, 1, head_size)
        self.v_cache = v
      else:
        self.k_cache = torch.cat((self.k_cache, k), dim=2) # (B, n_kv_heads, T, head_size)
        self.v_cache = torch.cat((self.v_cache, v), dim=2) # (B, n_kv_heads, T, head_size)
      self.k_cache = self.k_cache[:, :, -self.block_size:, :]
      self.v_cache = self.v_cache[:, :, -self.block_size:, :]
      k = self.k_cache
      v = self.v_cache

    k = k.repeat_interleave(self.n_groups, dim=1) # (B, n_kv_heads, T, head_size) => # (B, n_heads, T, head_size)
    v = v.repeat_interleave(self.n_groups, dim=1) # (B, n_heads, T, head_size)
    Tq, Tk = q.shape[2], k.shape[2] # num of tokens

    if self.flash:
      out = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0, is_causal=not use_cache)

    else:

      wei = (q @ k.transpose(-2,-1)) * (1.0 / math.sqrt(self.head_size)) # (B, n_heads, T, head_size) @ # (B, n_heads, head_size, Tk) => # (B, n_heads, T, Tk)
      if not use_cache:
        # training: apply mask
        wei = wei.masked_fill(self.tril[:,:,:Tq,:Tk] == 0, float('-inf'))

      # Training:
      # T  = full sequence length
      # Tk = full sequence length
      # T == Tk
      # wei: (B, n_heads, T, T)   ← square matrix

      # Decode:
      # T  = 1 ← only new token
      # Tk = cache length (all past tokens)
      # T != Tk
      # wei: (B, n_heads, 1, Tk)  ← single row

      wei = F.softmax(wei, dim=-1)
      wei = self.Dropout(wei)
      out = wei@v # (B, n_heads, T, Tk) @  (B, n_heads, Tk, head_size) => (B, n_heads, T, head_size)

    out = out.transpose(1,2).reshape(B, Tq, self.n_embd)
    out = self.out_proj(out)
    out = self.Dropout(out)
    return out

In [10]:
class Block(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.mha = MultiHeadAttention(config)
    self.ffwd = FeedForward(config)
    self.ln1 = RMSNorm(config)
    self.ln2 = RMSNorm(config)

  def clear_cache(self):
    self.mha.clear_cache()

  def forward(self, x, use_cache=False):
    x = x + self.mha(self.ln1(x), use_cache=use_cache)
    x = x + self.ffwd(self.ln2(x))
    return x

In [11]:
class Llama(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.vocab_size = config.vocab_size
    self.n_layer = config.n_layer
    self.n_embd = config.n_embd
    self.block_size = config.block_size

    self.token_embedding_table = nn.Embedding(self.vocab_size, self.n_embd)
    self.block = nn.ModuleList([Block(config) for _ in range(self.n_layer)])
    self.ln = RMSNorm(config)
    self.ffwd = nn.Linear(self.n_embd, self.vocab_size, bias=False)
    self.temperature = config.temperature

    # tie weights
    # self.ffwd.weight = self.token_embedding_table.weight

  def clear_cache(self):
    for block in self.block:
      block.clear_cache()

  def forward(self, idx, targets=None, use_cache=False):
    # idx = (B, T)
    x = self.token_embedding_table(idx) # (B, T, C)
    for block in self.block:
      x = block(x, use_cache=use_cache)
    x = self.ln(x)

    logits = self.ffwd(x) # (B, T, vocab_size)
    B, T, vocab_size = logits.shape
    if targets is None:
      loss = None
    else:
      loss = F.cross_entropy(logits.view(B*T, vocab_size), targets.view(B*T))
    return logits, loss

  def generate(self, idx, max_new_tokens):
    self.clear_cache()
    # idx = (B, T)

    for _ in range(max_new_tokens):
      x = self.token_embedding_table(idx[:, -1:])
      for block in self.block:
        x = block(x, use_cache=True)
      x = self.ln(x)
      logits = self.ffwd(x) # (B, 1, vocab_size)
      logits = logits[:, -1, :] # (B, vocab_size)

      # temperature
      logits = logits/self.temperature # (B, vocab_size)

      # top k
      top_k_values, top_k_indices = torch.topk(logits, k=50) # (B, 50)
      min_val = top_k_values[:, -1].unsqueeze(-1) # 50th value (B, ) => (B, 1)
      logits[logits < min_val] = float("-inf") # (B, vocab_size)

      probs = F.softmax(logits, dim=-1)
      idx_next = torch.multinomial(probs, num_samples=1)
      idx = torch.cat((idx, idx_next), dim=-1)
    return idx


In [12]:
from dataclasses import dataclass

@dataclass
class LlamaConfig:
    vocab_size: int = 85
    n_embd: int     = 128
    n_heads: int    = 4
    n_kv_heads: int = 2        # GQA: 2 KV heads, 4 Q heads per group
    n_layer: int    = 4
    block_size: int = 128     # max sequence length
    dropout: float  = 0.1
    eps: float      = 1e-6
    temperature: float = 0.4

### Data

In [16]:
from datasets import load_dataset

/Users/mehaksingal/Documents/cs_247B_deep_learning/dl_ucla/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
# load in streaming mode — doesn't download everything at once
ds = load_dataset("tatsu-lab/alpaca", streaming=True)

# iterate and collect only the output column
all_outputs = []
for row in ds["train"]:
    all_outputs.append(row["output"])

pretrain_text = "\n".join(all_outputs)
print(f"total chars: {len(pretrain_text):,}")

total chars: 14,108,783


In [18]:
import gc; gc.collect()

1749

In [19]:
from datasets import load_dataset
ds = load_dataset("tatsu-lab/alpaca")

# build vocab from output column only (clean english, no special chars)
all_text = "\n".join([row["output"] for row in ds["train"]])
vocab = sorted(set(all_text))
stoi = {ch: i for i, ch in enumerate(vocab)}
itos = {i: ch for i, ch in enumerate(vocab)}

encode = lambda s: [stoi[ch] for ch in s if ch in stoi]
decode = lambda t: "".join([itos[i] for i in t])

print(f"vocab size: {len(vocab)}")  # ~90 chars

Generating train split: 100%|██████████| 52002/52002 [00:00<00:00, 1715784.15 examples/s]


vocab size: 375


In [20]:
data = torch.tensor(encode(pretrain_text), dtype=torch.long)
print(f"total tokens: {len(data):,}")

# free the raw text — no longer needed
del pretrain_text, all_outputs
import gc; gc.collect()

n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

total tokens: 14,108,783


### Train

In [21]:
from dataclasses import dataclass

@dataclass
class LlamaConfig:
    vocab_size:  int   = len(vocab)  # from Alpaca outputs
    n_embd:      int   = 256
    n_heads:     int   = 8
    n_kv_heads:  int   = 4
    n_layer:     int   = 6
    block_size:  int   = 256         # Alpaca responses are longer than TinyStories
    dropout:     float = 0.1
    eps:         float = 1e-6
    temperature: float = 0.8

config = LlamaConfig()
print(f"vocab_size={config.vocab_size}, block_size={config.block_size}")

vocab_size=375, block_size=256


In [ ]:
def get_pretrain_batch(batch_size=16):
    ix = torch.randint(len(train_data) - config.block_size, (batch_size,))
    x  = torch.stack([train_data[i:i+config.block_size]   for i in ix])
    y  = torch.stack([train_data[i+1:i+config.block_size+1] for i in ix])
    return x.to(device), y.to(device)

model     = Llama(config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)

for step in range(10000):
    x, y = get_pretrain_batch()
    logits, loss = model(x, targets=y)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if step % 500 == 0:
        print(f"step {step:5d} | loss {loss.item():.4f}")

torch.save(model.state_dict(), "./llama_pretrain.pt")
print("done!")

In [23]:
torch.save(model.state_dict(), "./llama_pretrain.pt")
print("done!")

done!


In [24]:
import math

def get_lr(step, warmup=200, max_iters=10000, max_lr=1e-3, min_lr=1e-5):
    if step < warmup:
        return max_lr * step / warmup
    ratio = (step - warmup) / (max_iters - warmup)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * ratio))

model_cosine     = Llama(config).to(device)
optimizer_cosine = torch.optim.AdamW(model_cosine.parameters(), lr=1e-3, weight_decay=0.1)

for step in range(10000):
    lr = get_lr(step)
    for g in optimizer_cosine.param_groups:
        g["lr"] = lr

    x, y = get_pretrain_batch()
    logits, loss = model_cosine(x, targets=y)
    optimizer_cosine.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model_cosine.parameters(), 1.0)
    optimizer_cosine.step()

    if step % 500 == 0:
        print(f"step {step:5d} | lr {lr:.6f} | loss {loss.item():.4f}")

torch.save(model_cosine.state_dict(), "./llama_pretrain_cosine.pt")
print("done!")

step     0 | lr 0.000000 | loss 5.9467
step   500 | lr 0.000998 | loss 1.6398
step  1000 | lr 0.000984 | loss 1.3793
step  1500 | lr 0.000958 | loss 1.4649
step  2000 | lr 0.000920 | loss 1.3327
step  2500 | lr 0.000871 | loss 1.1611
step  3000 | lr 0.000814 | loss 1.2278
step  3500 | lr 0.000748 | loss 1.1622
step  4000 | lr 0.000676 | loss 1.2240
step  4500 | lr 0.000600 | loss 1.1740
step  5000 | lr 0.000521 | loss 1.1749
step  5500 | lr 0.000442 | loss 1.1252
step  6000 | lr 0.000364 | loss 1.0972
step  6500 | lr 0.000290 | loss 1.0933
step  7000 | lr 0.000222 | loss 1.0648
step  7500 | lr 0.000161 | loss 1.0734
step  8000 | lr 0.000108 | loss 1.0752
step  8500 | lr 0.000066 | loss 1.0303
step  9000 | lr 0.000035 | loss 1.0009
step  9500 | lr 0.000016 | loss 1.0811
done!


In [27]:
model.eval()
model.clear_cache()

# just start with a newline or first char in vocab
idx = torch.tensor(encode("\n"), dtype=torch.long).unsqueeze(0).to(device)

with torch.no_grad():
    out = model.generate(idx, max_new_tokens=200)

print(decode(out[0].tolist()))


7. Make advanced ethics and how to improve complex patterns
8. Promote management effects
9. Apply attention to provide pesticides to help automate teamwork or borrower
10. Regularly share in medical 


### SFT

In [49]:
import random

# use ALL 52k for SFT
sft_list = list(ds["train"])
random.shuffle(sft_list)
n         = int(0.9 * len(sft_list))
sft_train = sft_list[:n]   # ~47k
sft_val   = sft_list[n:]   # ~5k
print(f"sft train: {len(sft_train)}, sft val: {len(sft_val)}")

def format_sample(row):
    if row["input"]:
        prompt = f"Instruction: {row['instruction']}\nInput: {row['input']}\nResponse:\n"
    else:
        prompt = f"Instruction: {row['instruction']}\nResponse:\n"
    full = prompt + row["output"]
    return prompt, full

def get_sft_batch(data, batch_size=16):
    samples = random.sample(data, batch_size * 3)
    xs, ys = [], []
    for row in samples:
        prompt, full = format_sample(row)
        clean_prompt = "".join(ch if ch in stoi else " " for ch in prompt)
        clean_full   = "".join(ch if ch in stoi else " " for ch in full)
        full_ids   = encode(clean_full)
        prompt_len = len(encode(clean_prompt))
        if len(full_ids) < 2 or prompt_len >= len(full_ids):
            continue
        full_ids = full_ids[:config.block_size + 1]
        x = torch.tensor(full_ids[:-1], dtype=torch.long)
        y = torch.tensor(full_ids[1:],  dtype=torch.long)
        y[:prompt_len] = -100
        if (y != -100).sum() == 0:
            continue
        xs.append(x)
        ys.append(y)
        if len(xs) == batch_size:
            break
    if not xs:
        return get_sft_batch(data, batch_size)
    max_len = max(t.size(0) for t in xs)
    xs = torch.stack([F.pad(x, (0, max_len - x.size(0)), value=0)    for x in xs])
    ys = torch.stack([F.pad(y, (0, max_len - y.size(0)), value=-100) for y in ys])
    return xs.to(device), ys.to(device)

sft train: 46801, sft val: 5201


In [51]:
config = LlamaConfig(
    vocab_size  = len(vocab),
    n_embd      = 256,
    n_heads     = 8,
    n_kv_heads  = 4,
    n_layer     = 6,
    block_size  = 256,
    dropout     = 0.1,
    temperature = 0.8,
)

model = Llama(config).to(device)
model.load_state_dict(torch.load("./llama_pretrain.pt"))
model.train()

max_sft_iters = 5000
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

def get_sft_lr(step, max_iters=5000, max_lr=3e-4, min_lr=1e-6):
    ratio = step / max_iters
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * ratio))

smooth_loss = None

for step in range(max_sft_iters):
    lr = get_sft_lr(step)
    for g in optimizer.param_groups:
        g["lr"] = lr

    x, y = get_sft_batch(sft_train)
    logits, _ = model(x)
    B, T, V = logits.shape
    loss = F.cross_entropy(logits.view(B*T, V), y.view(B*T), ignore_index=-100)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    smooth_loss = loss.item() if smooth_loss is None else 0.95*smooth_loss + 0.05*loss.item()

    if step % 200 == 0:
        print(f"step {step:5d} | lr {lr:.6f} | smooth {smooth_loss:.4f}")

torch.save(model.state_dict(), "./llama_sft_final.pt")
print("done!")

step     0 | lr 0.000300 | smooth 1.0889
step   200 | lr 0.000299 | smooth 1.1108
step   400 | lr 0.000295 | smooth 1.0615
step   600 | lr 0.000290 | smooth 1.0944
step   800 | lr 0.000282 | smooth 1.0389
step  1000 | lr 0.000271 | smooth 1.0397
step  1200 | lr 0.000259 | smooth 1.0541
step  1400 | lr 0.000246 | smooth 1.0487
step  1600 | lr 0.000231 | smooth 1.0320
step  1800 | lr 0.000214 | smooth 1.0309
step  2000 | lr 0.000197 | smooth 1.0372
step  2200 | lr 0.000179 | smooth 1.0295
step  2400 | lr 0.000160 | smooth 0.9966
step  2600 | lr 0.000141 | smooth 1.0081
step  2800 | lr 0.000122 | smooth 1.0155
step  3000 | lr 0.000104 | smooth 1.0026
step  3200 | lr 0.000087 | smooth 1.0105
step  3400 | lr 0.000070 | smooth 0.9948
step  3600 | lr 0.000055 | smooth 0.9891
step  3800 | lr 0.000042 | smooth 0.9956
step  4000 | lr 0.000030 | smooth 0.9949
step  4200 | lr 0.000019 | smooth 0.9909
step  4400 | lr 0.000011 | smooth 0.9967
step  4600 | lr 0.000006 | smooth 0.9929
step  4800 | lr 

In [ ]:
torch.save(model.state_dict(), "./llama_cosine_ft.pt")
print("SFT done!")

SFT done!


In [67]:
model.eval()
model.clear_cache()

model.temperature = 0.8
prompt = "Write some code.\nResponse:\n"
idx = torch.tensor(encode(prompt), dtype=torch.long).unsqueeze(0).to(device)

with torch.no_grad():
    out = model.generate(idx, max_new_tokens=200)

print(decode(out[0].tolist()))

Write some code.
Response:
                        

 
                     array = 1
                      return 2
        for n in list
               binary = 1:
             print("A + 1
    return False
    for max in lis
